# Partie 6 — Pipeline de production

Projet Écosystème Big Data 2 — FastFood.ma — EMSI 5ème année, 2026-2027.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = Path("/work")
sys.path[:0] = [str(root / "src"), str(root / "third_party")]


Silver : dédoublonnage `(order_id, status)` sous le watermark, jointure stream-static avec les restaurants, colonne `distance_band`, Parquet partitionné par `city` et `order_date`.

Or : fenêtres tumbling de 10 minutes, commandes `DELIVERED`, compte et chiffre d'affaires, **Delta Lake** en mode `append`. Une fenêtre n'est écrite qu'une fois close, donc environ 20 minutes d'event-time après son début (fenêtre 10 min + watermark 10 min). Le sink `gold-live` est en `update` : il affiche les chiffres avant la fermeture de la fenêtre. Il n'est pas exactly-once.

Annulations : fenêtre glissante de 10 minutes, tous les 5 minutes.
Sessions livreur : `session_window` de 5 minutes d'inactivité. `worked_seconds` retire ce gap, sinon la durée affiche 5 minutes de pause.

Pour fermer les fenêtres sans attendre 20 minutes, dans un autre terminal :

```bash
make replay
```

Ça republie les 7 jours d'historique produits par le générateur du sujet. Le watermark avance vite et les fenêtres se ferment.


In [ ]:
from fastfood.partie6 import preview_batch, run
from fastfood.session import build_spark

spark = build_spark("fastfood-partie6-production", kafka=True, delta=True)
preview_batch(spark)


In [ ]:
# Le producteur Kafka doit déjà tourner.
run(spark, seconds=45)


## Exactly-once, checkpoint, consumer group

Trois conditions, toutes nécessaires :

1. La source est rejouable. Kafka garde les messages, on relit à partir d'un offset.
2. Le checkpoint (`lake/checkpoints/gold_kpi/offsets`) mémorise le dernier offset **commité avec** le micro-batch.
3. Le sink est transactionnel. Delta écrit une version. Si le job meurt avant le commit, le batch est rejoué. S'il a commité, il ne l'est pas une seconde fois.

La console n'a pas de commit. Un redémarrage réaffiche les mêmes lignes. C'est pour ça que l'or n'est pas un `format("console")`.

`kafka-consumer-groups.sh --describe` montre le groupe `fastfood-ma-…`. Spark ne s'appuie pas sur les offsets committés dans Kafka pour reprendre : le fichier de checkpoint fait foi. Les deux sorties se comparent, elles ne se remplacent pas.

Reprise : on note le nombre de lignes Delta, on arrête la requête, on relance avec le même checkpoint. Sans événement nouveau le compte ne bouge pas. Si le producteur a continué, les anciennes fenêtres restent et les nouvelles s'ajoutent. `count(*)` reste égal à `count(distinct city, window_start)`.


In [ ]:
import json
from pathlib import Path

from fastfood.partie6 import prove_restart

proof = prove_restart(spark)
destination = Path("outputs")
if not destination.exists():
    destination = Path("/work/outputs")
destination.mkdir(parents=True, exist_ok=True)
(destination / "reprise.json").write_text(json.dumps(proof, indent=2) + "\n", encoding="utf-8")
proof


In [ ]:
from fastfood.partie6 import describe_gold, print_offsets

describe_gold(spark)
print_offsets(spark, "gold_kpi")
